In [1]:
import requests
import pandas as pd

In [2]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 17.3850,
    "longitude": 78.4867,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "pressure_msl",
        "wind_speed_10m",
        "weather_code"
    ],
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum"
    ],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)
response.raise_for_status()

weather_data = response.json()

In [3]:
weather_data.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly', 'daily_units', 'daily'])

In [4]:
hourly_df = pd.DataFrame(weather_data["hourly"])
hourly_df["time"] = pd.to_datetime(hourly_df["time"])

hourly_df.head()

,time,temperature_2m,relative_humidity_2m,precipitation,pressure_msl,wind_speed_10m,weather_code
0,2022-01-01 00:00:00,19.4,77,0.0,1019.1,8.2,3
1,2022-01-01 01:00:00,20.4,72,0.0,1018.6,9.2,3
2,2022-01-01 02:00:00,20.0,74,0.0,1018.2,8.4,3
3,2022-01-01 03:00:00,19.6,76,0.0,1018.2,9.4,1
4,2022-01-01 04:00:00,18.4,82,0.0,1018.0,9.9,0


In [8]:
hourly_df.isnull().sum()

time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
pressure_msl            0
wind_speed_10m          0
weather_code            0
dtype: int64

In [5]:
daily_df = pd.DataFrame(weather_data["daily"])
daily_df["time"] = pd.to_datetime(daily_df["time"])

daily_df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum
0,2022-01-01,27.8,16.9,0.0
1,2022-01-02,27.1,16.9,0.0
2,2022-01-03,27.3,15.3,0.0
3,2022-01-04,27.0,14.9,0.0
4,2022-01-05,27.6,14.8,0.0


In [9]:
daily_df.isnull().sum()

time                  0
temperature_2m_max    0
temperature_2m_min    0
precipitation_sum     0
dtype: int64

In [6]:
print("Hourly shape:", hourly_df.shape)
print("Daily shape:", daily_df.shape)

Hourly shape: (35064, 7)
Daily shape: (1461, 4)


In [10]:
hourly_df.info()
daily_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35064 entries, 0 to 35063
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  35064 non-null  datetime64[us]
 1   temperature_2m        35064 non-null  float64       
 2   relative_humidity_2m  35064 non-null  int64         
 3   precipitation         35064 non-null  float64       
 4   pressure_msl          35064 non-null  float64       
 5   wind_speed_10m        35064 non-null  float64       
 6   weather_code          35064 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(2)
memory usage: 1.9 MB
<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   time                1461 non-null   datetime64[us]
 1   temperature_2m_max  1461 non-null   float64 

In [11]:
hourly_df.to_csv(
    "../../data/raw/raw_weather_hourly.csv",
    index=False
)

daily_df.to_csv(
    "../../data/raw/raw_weather_daily.csv",
    index=False
)

# Weather Feature Engineering

In [12]:
import pandas as pd
import numpy as np

In [13]:
hourly_df = pd.read_csv(
    "../../data/raw/raw_weather_hourly.csv",
    parse_dates=["time"]
)

daily_df = pd.read_csv(
    "../../data/raw/raw_weather_daily.csv",
    parse_dates=["time"]
)

In [14]:
hourly_df.head()

,time,temperature_2m,relative_humidity_2m,precipitation,pressure_msl,wind_speed_10m,weather_code
0,2022-01-01 00:00:00,19.4,77,0.0,1019.1,8.2,3
1,2022-01-01 01:00:00,20.4,72,0.0,1018.6,9.2,3
2,2022-01-01 02:00:00,20.0,74,0.0,1018.2,8.4,3
3,2022-01-01 03:00:00,19.6,76,0.0,1018.2,9.4,1
4,2022-01-01 04:00:00,18.4,82,0.0,1018.0,9.9,0


In [15]:
daily_df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum
0,2022-01-01,27.8,16.9,0.0
1,2022-01-02,27.1,16.9,0.0
2,2022-01-03,27.3,15.3,0.0
3,2022-01-04,27.0,14.9,0.0
4,2022-01-05,27.6,14.8,0.0


In [16]:
hourly_df["date"] = hourly_df["time"].dt.date

In [17]:
hourly_daily = (
    hourly_df.groupby("date")
    .agg(
        temperature_mean=("temperature_2m", "mean"),
        humidity_mean=("relative_humidity_2m", "mean"),
        pressure_mean=("pressure_msl", "mean"),
        wind_speed_mean=("wind_speed_10m", "mean"),
        precipitation_hourly_sum=("precipitation", "sum"),
    )
    .reset_index()
)

In [18]:
hourly_daily["date"] = pd.to_datetime(hourly_daily["date"])

In [19]:
hourly_daily.head()

,date,temperature_mean,humidity_mean,pressure_mean,wind_speed_mean,precipitation_hourly_sum
0,2022-01-01,21.970833,66.125000,1018.662500,8.916667,0.0
1,2022-01-02,21.362500,65.875000,1017.191667,7.429167,0.0
2,2022-01-03,20.662500,62.833333,1016.412500,7.575000,0.0
3,2022-01-04,20.391667,63.833333,1016.416667,8.654167,0.0
4,2022-01-05,20.762500,69.166667,1016.112500,7.720833,0.0


In [20]:
daily_df = daily_df.rename(columns={"time": "date"})

In [21]:
daily_df.head()

,date,temperature_2m_max,temperature_2m_min,precipitation_sum
0,2022-01-01,27.8,16.9,0.0
1,2022-01-02,27.1,16.9,0.0
2,2022-01-03,27.3,15.3,0.0
3,2022-01-04,27.0,14.9,0.0
4,2022-01-05,27.6,14.8,0.0


In [22]:
weather_df = pd.merge(
    daily_df,
    hourly_daily,
    on="date",
    how="inner"
)

In [23]:
weather_df.head()

,date,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_mean,humidity_mean,pressure_mean,wind_speed_mean,precipitation_hourly_sum
0,2022-01-01,27.8,16.9,0.0,21.970833,66.125000,1018.662500,8.916667,0.0
1,2022-01-02,27.1,16.9,0.0,21.362500,65.875000,1017.191667,7.429167,0.0
2,2022-01-03,27.3,15.3,0.0,20.662500,62.833333,1016.412500,7.575000,0.0
3,2022-01-04,27.0,14.9,0.0,20.391667,63.833333,1016.416667,8.654167,0.0
4,2022-01-05,27.6,14.8,0.0,20.762500,69.166667,1016.112500,7.720833,0.0


In [24]:
weather_df.shape

(1461, 9)

In [25]:
weather_df.isnull().sum()

date                        0
temperature_2m_max          0
temperature_2m_min          0
precipitation_sum           0
temperature_mean            0
humidity_mean               0
pressure_mean               0
wind_speed_mean             0
precipitation_hourly_sum    0
dtype: int64

In [26]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      1461 non-null   datetime64[us]
 1   temperature_2m_max        1461 non-null   float64       
 2   temperature_2m_min        1461 non-null   float64       
 3   precipitation_sum         1461 non-null   float64       
 4   temperature_mean          1461 non-null   float64       
 5   humidity_mean             1461 non-null   float64       
 6   pressure_mean             1461 non-null   float64       
 7   wind_speed_mean           1461 non-null   float64       
 8   precipitation_hourly_sum  1461 non-null   float64       
dtypes: datetime64[us](1), float64(8)
memory usage: 102.9 KB


In [27]:
weather_df = weather_df.sort_values("date").reset_index(drop=True)

In [28]:
weather_df["temperature_lag_1"] = (
    weather_df["temperature_2m_max"].shift(1)
)

In [29]:
weather_df["temperature_lag_3"] = (
    weather_df["temperature_2m_max"].shift(3)
)

In [30]:
weather_df["temperature_lag_7"] = (
    weather_df["temperature_2m_max"].shift(7)
)

In [31]:
weather_df["temperature_rolling_3"] = (
    weather_df["temperature_2m_max"]
    .rolling(window=3)
    .mean()
)

In [32]:
weather_df["temperature_rolling_7"] = (
    weather_df["temperature_2m_max"]
    .rolling(window=7)
    .mean()
)

In [33]:
weather_df["humidity_rolling_3"] = (
    weather_df["humidity_mean"]
    .rolling(window=3)
    .mean()
)

In [34]:
weather_df["pressure_rolling_3"] = (
    weather_df["pressure_mean"]
    .rolling(window=3)
    .mean()
)

In [35]:
weather_df["day_of_week"] = weather_df["date"].dt.dayofweek

In [36]:
weather_df["month"] = weather_df["date"].dt.month

In [37]:
weather_df["target_next_day_max_temp"] = (
    weather_df["temperature_2m_max"].shift(-1)
)

In [38]:
weather_df.head(10)

,date,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_mean,humidity_mean,pressure_mean,wind_speed_mean,precipitation_hourly_sum,temperature_lag_1,temperature_lag_3,temperature_lag_7,temperature_rolling_3,temperature_rolling_7,humidity_rolling_3,pressure_rolling_3,day_of_week,month,target_next_day_max_temp
0,2022-01-01,27.8,16.9,0.0,21.970833,66.125000,1018.662500,8.916667,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,1,27.1
1,2022-01-02,27.1,16.9,0.0,21.362500,65.875000,1017.191667,7.429167,0.0,27.8,NaN,NaN,NaN,NaN,NaN,NaN,6,1,27.3
2,2022-01-03,27.3,15.3,0.0,20.662500,62.833333,1016.412500,7.575000,0.0,27.1,NaN,NaN,27.400000,NaN,64.944444,1017.422222,0,1,27.0
3,2022-01-04,27.0,14.9,0.0,20.391667,63.833333,1016.416667,8.654167,0.0,27.3,27.8,NaN,27.133333,NaN,64.180556,1016.673611,1,1,27.6
4,2022-01-05,27.6,14.8,0.0,20.762500,69.166667,1016.112500,7.720833,0.0,27.0,27.1,NaN,27.300000,NaN,65.277778,1016.313889,2,1,27.5
5,2022-01-06,27.5,14.7,0.0,20.875000,67.416667,1015.020833,9.358333,0.0,27.6,27.3,NaN,27.366667,NaN,66.805556,1015.850000,3,1,27.6
6,2022-01-07,27.6,16.9,0.0,21.725000,68.416667,1014.962500,10.562500,0.0,27.5,27.0,NaN,27.566667,27.414286,68.333333,1015.365278,4,1,27.6
7,2022-01-08,27.6,18.0,0.3,22.000000,72.750000,1015.225000,11.425000,0.3,27.6,27.6,27.8,27.566667,27.385714,69.527778,1015.069444,5,1,28.7
8,2022-01-09,28.7,18.0,0.4,22.508333,70.583333,1013.383333,8.841667,0.4,27.6,27.5,27.1,27.966667,27.614286,70.583333,1014.523611,6,1,26.5
9,2022-01-10,26.5,18.2,3.0,21.400000,79.625000,1013.962500,7.333333,3.0,28.7,27.6,27.3,27.600000,27.500000,74.319444,1014.190278,0,1,26.6


In [39]:
weather_df.columns

Index(['date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'temperature_mean', 'humidity_mean', 'pressure_mean', 'wind_speed_mean',
       'precipitation_hourly_sum', 'temperature_lag_1', 'temperature_lag_3',
       'temperature_lag_7', 'temperature_rolling_3', 'temperature_rolling_7',
       'humidity_rolling_3', 'pressure_rolling_3', 'day_of_week', 'month',
       'target_next_day_max_temp'],
      dtype='str')

In [40]:
weather_df.shape

(1461, 19)

In [41]:
weather_df.isnull().sum()

date                        0
temperature_2m_max          0
temperature_2m_min          0
precipitation_sum           0
temperature_mean            0
humidity_mean               0
pressure_mean               0
wind_speed_mean             0
precipitation_hourly_sum    0
temperature_lag_1           1
temperature_lag_3           3
temperature_lag_7           7
temperature_rolling_3       2
temperature_rolling_7       6
humidity_rolling_3          2
pressure_rolling_3          2
day_of_week                 0
month                       0
target_next_day_max_temp    1
dtype: int64

In [42]:
weather_df = weather_df.dropna().reset_index(drop=True)

In [43]:
weather_df.isnull().sum()

date                        0
temperature_2m_max          0
temperature_2m_min          0
precipitation_sum           0
temperature_mean            0
humidity_mean               0
pressure_mean               0
wind_speed_mean             0
precipitation_hourly_sum    0
temperature_lag_1           0
temperature_lag_3           0
temperature_lag_7           0
temperature_rolling_3       0
temperature_rolling_7       0
humidity_rolling_3          0
pressure_rolling_3          0
day_of_week                 0
month                       0
target_next_day_max_temp    0
dtype: int64

In [44]:
weather_df.shape

(1453, 19)

In [45]:
weather_df.head()

,date,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_mean,humidity_mean,pressure_mean,wind_speed_mean,precipitation_hourly_sum,temperature_lag_1,temperature_lag_3,temperature_lag_7,temperature_rolling_3,temperature_rolling_7,humidity_rolling_3,pressure_rolling_3,day_of_week,month,target_next_day_max_temp
0,2022-01-08,27.6,18.0,0.3,22.000000,72.750000,1015.225000,11.425000,0.3,27.6,27.6,27.8,27.566667,27.385714,69.527778,1015.069444,5,1,28.7
1,2022-01-09,28.7,18.0,0.4,22.508333,70.583333,1013.383333,8.841667,0.4,27.6,27.5,27.1,27.966667,27.614286,70.583333,1014.523611,6,1,26.5
2,2022-01-10,26.5,18.2,3.0,21.400000,79.625000,1013.962500,7.333333,3.0,28.7,27.6,27.3,27.600000,27.500000,74.319444,1014.190278,0,1,26.6
3,2022-01-11,26.6,17.3,4.9,20.820833,82.958333,1013.758333,10.608333,4.9,26.5,27.6,27.0,27.266667,27.442857,77.722222,1013.701389,1,1,26.1
4,2022-01-12,26.1,17.0,1.5,20.275000,83.583333,1013.520833,8.545833,1.5,26.6,28.7,27.6,26.400000,27.228571,82.055556,1013.747222,2,1,24.1


In [46]:
weather_df.tail()

,date,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_mean,humidity_mean,pressure_mean,wind_speed_mean,precipitation_hourly_sum,temperature_lag_1,temperature_lag_3,temperature_lag_7,temperature_rolling_3,temperature_rolling_7,humidity_rolling_3,pressure_rolling_3,day_of_week,month,target_next_day_max_temp
1448,2025-12-26,26.6,16.2,0.0,21.425000,51.833333,1014.975000,4.858333,0.0,26.8,27.5,27.5,26.833333,27.114286,48.902778,1014.695833,4,12,26.9
1449,2025-12-27,26.9,16.2,0.0,21.483333,52.708333,1015.270833,4.283333,0.0,26.6,27.1,27.3,26.766667,27.057143,50.541667,1015.062500,5,12,26.8
1450,2025-12-28,26.8,16.2,0.0,21.504167,51.916667,1015.795833,4.920833,0.0,26.9,26.8,27.0,26.766667,27.028571,52.152778,1015.347222,6,12,26.6
1451,2025-12-29,26.6,16.5,0.0,21.466667,50.833333,1015.516667,5.387500,0.0,26.8,26.6,27.5,26.766667,26.900000,51.819444,1015.527778,0,12,28.0
1452,2025-12-30,28.0,16.1,0.0,21.995833,51.500000,1014.333333,2.775000,0.0,26.6,26.9,27.5,27.133333,26.971429,51.416667,1015.215278,1,12,28.3


In [47]:
weather_df.to_csv(
    "../../data/processed/weather_features.csv",
    index=False
)

In [48]:
print(weather_df.shape)
print(weather_df.isnull().sum())
print(weather_df["date"].min())
print(weather_df["date"].max())
weather_df.describe()

(1453, 19)
date                        0
temperature_2m_max          0
temperature_2m_min          0
precipitation_sum           0
temperature_mean            0
humidity_mean               0
pressure_mean               0
wind_speed_mean             0
precipitation_hourly_sum    0
temperature_lag_1           0
temperature_lag_3           0
temperature_lag_7           0
temperature_rolling_3       0
temperature_rolling_7       0
humidity_rolling_3          0
pressure_rolling_3          0
day_of_week                 0
month                       0
target_next_day_max_temp    0
dtype: int64
2022-01-08 00:00:00
2025-12-30 00:00:00


,date,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_mean,humidity_mean,pressure_mean,wind_speed_mean,precipitation_hourly_sum,temperature_lag_1,temperature_lag_3,temperature_lag_7,temperature_rolling_3,temperature_rolling_7,humidity_rolling_3,pressure_rolling_3,day_of_week,month,target_next_day_max_temp
count,1453,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000,1453.000000
mean,2024-01-04 00:00:00,31.233930,21.451686,2.773503,25.955557,62.883287,1009.626382,10.023905,2.773503,31.233655,31.234825,31.236063,31.233953,31.234648,62.894854,1009.626557,3.000000,6.545767,31.234412
min,2022-01-08 00:00:00,23.100000,11.100000,0.000000,18.716667,21.541667,999.425000,2.775000,0.000000,23.100000,23.100000,23.100000,23.600000,24.400000,24.611111,999.865278,0.000000,1.000000,23.100000
25%,2023-01-06 00:00:00,28.800000,19.200000,0.000000,23.970833,48.875000,1006.383333,6.866667,0.000000,28.800000,28.800000,28.800000,28.766667,28.642857,48.902778,1006.391667,1.000000,4.000000,28.800000
50%,2024-01-04 00:00:00,30.200000,22.200000,0.000000,25.529167,63.875000,1009.395833,8.570833,0.000000,30.200000,30.200000,30.200000,30.166667,30.071429,63.958333,1009.305556,3.000000,7.000000,30.200000
75%,2025-01-01 00:00:00,33.400000,23.400000,2.100000,27.625000,77.500000,1013.291667,11.937500,2.100000,33.400000,33.400000,33.400000,33.366667,33.557143,77.680556,1013.266667,5.000000,10.000000,33.400000
max,2025-12-30 00:00:00,41.400000,28.900000,82.700000,33.808333,93.500000,1018.933333,31.050000,82.700000,41.400000,41.400000,41.400000,40.733333,40.371429,91.625000,1018.227778,6.000000,12.000000,41.400000
std,NaN,3.511953,2.978401,6.495614,2.923834,17.065516,4.196680,4.561715,6.495614,3.512222,3.510851,3.509406,3.420103,3.322770,16.586919,4.136092,2.002409,3.434973,3.511502


In [49]:
weather_df["date"].is_monotonic_increasing

True

In [50]:
weather_df[["temperature_2m_max", "target_next_day_max_temp"]].head(10)

,temperature_2m_max,target_next_day_max_temp
0,27.6,28.7
1,28.7,26.5
2,26.5,26.6
3,26.6,26.1
4,26.1,24.1
5,24.1,27.3
6,27.3,25.2
7,25.2,26.5
8,26.5,24.9
9,24.9,27.0
